# Q54: Handling Missing Data
**Topic:** Classical-ml | **Difficulty:** Easy | **Time:** 10 min
**Sheet:** Grind75ML

---

## Question
How do you handle missing data?

## Detailed Answer

### Types of Missingness
| Type | Meaning | Example |
|------|---------|--------|
| MCAR | Missing completely at random | Random sensor failure |
| MAR | Missing depends on observed data | Higher income → less likely to report |
| MNAR | Missing depends on unobserved data | Depressed people less likely to fill survey |

### Strategies

#### 1. Deletion
- **Listwise**: Drop rows with any missing value
  - OK if MCAR and <5% missing
  - Loses data, biased if not MCAR
- **Column deletion**: Drop feature with >50% missing

#### 2. Imputation
- **Mean/Median/Mode**: Simple, fast, distorts distribution
- **KNN imputation**: Use K nearest neighbors to estimate
- **Iterative (MICE)**: Model each missing feature as a function of others
- **Constant**: Fill with sentinel value (e.g., -999)

#### 3. Model-Based
- **XGBoost/LightGBM**: Handle missing values natively (learns optimal split direction)
- **Deep learning**: Masking or learned embeddings for missing

#### 4. Indicator Feature
- Add a binary column `is_missing` for each imputed feature
- Lets the model learn that missingness itself is informative

### Guidelines
| Scenario | Recommendation |
|----------|---------------|
| <5% missing, MCAR | Drop rows or mean impute |
| 5-20% missing | KNN or MICE imputation + indicator |
| >50% missing column | Drop column (unless informative) |
| Tree-based model | Use native handling (XGBoost) |
| Production | Impute + add is_missing indicator |

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Create data with missing values
np.random.seed(42)
df = pd.DataFrame(np.random.randn(100, 4), columns=['A', 'B', 'C', 'D'])
mask = np.random.random(df.shape) < 0.15
df[mask] = np.nan
print(f'Missing values:\n{df.isnull().sum()}\n')

# Mean imputation
mean_imp = SimpleImputer(strategy='mean').fit_transform(df)
print(f'Mean imputed shape: {mean_imp.shape}')

# KNN imputation
knn_imp = KNNImputer(n_neighbors=5).fit_transform(df)
print(f'KNN imputed shape: {knn_imp.shape}')

# MICE (Iterative) imputation
mice_imp = IterativeImputer(max_iter=10, random_state=42).fit_transform(df)
print(f'MICE imputed shape: {mice_imp.shape}')

## Key Takeaways
- **KNN/MICE imputation** is better than mean/median — preserves relationships
- Always add **is_missing indicator** columns in production
- **XGBoost** handles missing values natively — often the simplest solution
- Understand missingness type (MCAR/MAR/MNAR) before choosing strategy